# Non-Gaussian Outcomes

Many outcomes are not continuous: whether a business reopened after a disaster, the number of homicides in a county. The spatial processes carry over, but the model needs a link between the process and the outcome, and the coefficients need translating back to the outcome's own scale before they mean anything. The classes follow the same pattern as before: a formula, a graph, and `fit()`.

In [ ]:
import arviz as az
import geopandas as gpd
import numpy as np
import pandas as pd
from libpysal.graph import Graph
from neighbayes.models import SARLogit, SARNegBin, SARProbit

az.rcParams["stats.ci_prob"] = 0.95
fit_kwargs = dict(draws=1000, tune=1000, chains=4, random_seed=42)

## Binary outcomes: reopening after Hurricane Katrina

The data record 673 businesses on three streets in New Orleans and whether each reopened within three months of the hurricane (`y1`). The example follows LeSage et al. (2011), by way of Luc Anselin's course material. The covariates are the depth of flooding (`flood_dpth`), the log of median income (`log_medinc`), indicators for small and large businesses (`small_size`, `large_size`, against medium), for low- and high-status customers (`low_cust`, `high_cust`), for sole proprietorships (`sole_prop`), and for national chains (`nat_chain`).

In [ ]:
katrina = gpd.read_file("data/Katrina/Katrina_utm.shp")
katrina["y1"].mean()

Just under half of the businesses reopened within three months.

Following LeSage et al., each business's neighbors are its 11 nearest. Some businesses share an address, and nearest neighbors are undefined among points at the same location, so `Graph.build_knn` refuses to build the graph unless we say how to treat them. `coplanar="clique"` connects businesses at the same address to one another.

In [ ]:
graph = Graph.build_knn(katrina, k=11, coplanar="clique").transform("r")
graph.cardinalities.describe()

### The spatial probit

The probit model treats reopening as the visible sign of a latent propensity, $y_i^*$: a business reopens when its propensity is positive. The spatial probit lets the propensities depend on one another,

$$y^* = \rho W y^* + X\beta + \varepsilon, \qquad \varepsilon \sim N(0, I), \qquad y_i = 1[y_i^* > 0],$$

so a business's willingness to reopen moves with that of the businesses around it. The Gibbs sampler draws the latent propensities along with the parameters.

In [ ]:
formula = (
    "y1 ~ flood_dpth + log_medinc + small_size + large_size"
    " + low_cust + high_cust + sole_prop + nat_chain"
)

probit = SARProbit(formula=formula, data=katrina, W=graph)
probit.fit(**fit_kwargs)
probit.summary(round_to=3)

The dependence is strong: $\rho$ is close to one half. Deeper flooding and low-status customers lower the propensity to reopen, and sole proprietors were more likely to reopen than other businesses.

### Effects on the probability scale

A probit coefficient is measured on the scale of the latent propensity, whose variance is fixed at one by assumption, so its size means little on its own. With `scale="probability"`, `impacts()` reports effects on the probability of reopening instead: the change in that probability, in a business and among its neighbors, for a unit change in a covariate.

In [ ]:
probit_effects = probit.impacts(scale="probability")
probit_effects[["direct", "indirect", "total", "total_ci_lower", "total_ci_upper"]].round(3)

A unit of additional flood depth lowers the probability that a business reopens within three months by about five percentage points directly, and by about four more through its effect on the businesses around it. Nearly half of the damage the flood did to a business's prospects came through its neighbors.

### The spatial logit

The logit differs from the probit only in the distribution of $\varepsilon$, logistic rather than normal, and in `neighbayes` it differs by one word.

In [ ]:
logit = SARLogit(formula=formula, data=katrina, W=graph)
logit.fit(**fit_kwargs)
logit.summary(round_to=3)

The logit coefficients are larger than the probit's, roughly by the factor that separates the standard deviations of the two distributions ($\pi/\sqrt{3} \approx 1.8$ against one). That difference is a matter of scale, not substance, and it disappears on the probability scale.

In [ ]:
logit_effects = logit.impacts(scale="probability")
pd.concat(
    {"probit": probit_effects["total"], "logit": logit_effects["total"]}, axis=1
).round(3)

The two models agree on the total effects, and on $\rho$. The choice between them matters less than the choice to report effects on a scale a reader can interpret.

## Count outcomes: homicides in U.S. counties

The second example uses homicides in the 3,085 counties of the continental United States in 1990, the data of Baller et al. (2001). The file records homicides as a three-year average centered on the census year, so we multiply by three to recover the three-year count. The covariates are an index of resource deprivation (`RD90`), the unemployment rate (`UE90`), the divorce rate (`DV90`), the median age (`MA90`), and an indicator for the South (`SOUTH`).

In [ ]:
nat = gpd.read_file("data/NAT/NAT.shp")
nat["homicides"] = (nat["HC90"] * 3).round().astype(int)
nat["homicides"].describe()

In [ ]:
nat["homicides"].var() / nat["homicides"].mean()

A Poisson model would require the variance of the counts to equal their mean; here it is more than a thousand times larger. The negative binomial adds a dispersion parameter, `alpha`, to absorb the difference.

Counties also differ in population by more than four orders of magnitude, so the counts need an exposure term. The usual device is an offset, $\log(\text{population})$ entered with its coefficient fixed at one, which makes the model one of homicide rates. `neighbayes` has no offset, so we enter log population as a covariate and let the data estimate its coefficient.

In [ ]:
nat["log_pop"] = np.log(nat["PO90"])
nat_graph = Graph.build_contiguity(nat, rook=False).transform("r")

negbin = SARNegBin(
    formula="homicides ~ log_pop + RD90 + UE90 + DV90 + MA90 + SOUTH", data=nat, W=nat_graph
)
negbin.fit(**fit_kwargs)
negbin.summary(round_to=3)

The coefficient on log population is clearly above one: homicides rise faster than population, so a county with twice the population expects more than twice the homicides. An offset would have imposed proportionality and hidden that, which echoes the superlinear scaling of crime with city size (Bettencourt et al. 2007). Resource deprivation and the South raise the expected count, and $\rho$ is positive.

This model mixes more slowly than the ones before it: some `r_hat` values sit near the 1.01 limit from the first notebook of the afternoon, and effective sample sizes fall to the hundreds. For a paper we would run it longer. Here, doubling the run leaves the estimates where they are.

The default effects are on the scale of the log of the expected count, so they read as proportional changes; `scale="count"` reports them in expected homicides.

In [ ]:
negbin.impacts(scale="count")[["direct", "indirect", "total", "total_ci_lower", "total_ci_upper"]].round(2)

In [ ]:
effects = negbin.impacts()
(effects["indirect"] / effects["direct"]).round(3)

The ratio of indirect to direct effect is the same for every covariate. This is the restriction of the spatial lag model from the previous notebook, and it holds for the count model too: the lag on the outcome makes every covariate spill over in the same proportion.

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
The Katrina data also record whether each business reopened within six months (<code>y2</code>) and within twelve (<code>y3</code>). Refit the spatial probit for each. How do $\rho$ and the effect of flood depth change as time passes?
</div>

In [ ]:
# %load solutions/09.py